<h1 align="center">TÉCNICO EM CIÊNCIA DE DADOS</h1>

<h2 align="center">Colocando em prática — Padronização e organização dos dados</h2>

**Componente:** Análise Exploratória de Dados e Inteligência de Negócios  
**Unidade Curricular:** Análise Exploratória de Dados  
**Tema da Semana:** Transformação e Preparação de Dados  
**Semana 9 — Aula 2:** Padronização e organização dos dados


# Situação profissional

Você integra a equipe de dados de uma rede de papelarias. Registros vindos de fontes diferentes precisam ser preparados para que possam ser usados de forma consistente na análise.

Nesta atividade, você vai trabalhar com **três situações diferentes**:

1. padronizar formatos em dados reais da base integrada;
2. unificar rótulos que representam a mesma categoria em um recorte didático;
3. reorganizar valores que estão incorporados a um campo de texto.

O código já está preparado. Seu trabalho é **identificar o problema, aplicar o critério indicado e conferir o resultado**.


## Preparação

Use os arquivos:

- `DADOS3EMC3B2S9A2_aluno.ipynb`
- `rede_papelarias_B2_integrado.csv`

Mantenha os dois na mesma pasta. Execute as células na ordem em que aparecem.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ARQUIVO = Path("rede_papelarias_B2_integrado.csv")

if not ARQUIVO.exists():
    raise FileNotFoundError(
        "Coloque 'rede_papelarias_B2_integrado.csv' na mesma pasta deste notebook."
    )

dados_brutos = pd.read_csv(
    ARQUIVO,
    dtype={
        "despesas_com_pessoal": "string",
        "outras_despesas_operacionais": "string",
    },
)

# Trabalhamos em uma cópia e recompomos a retirada da cópia integral já identificada no projeto.
dados = dados_brutos.drop_duplicates().copy()

print(f"Registros na base de trabalho: {len(dados)}")


# Situação 1 — Formatos diferentes para despesas

Na base integrada, duas variáveis de despesas contêm registros equivalentes escritos em formatos diferentes. Nesta situação, o objetivo é **uniformizar a forma de registro**, preservando a informação representada.

Observe alguns registros que fogem ao padrão predominante.


In [ ]:
amostra_pessoal = (
    dados.loc[
        dados["despesas_com_pessoal"].str.contains(r"R\$|,", regex=True, na=False),
        ["id_registro", "despesas_com_pessoal"]
    ]
    .head(4)
    .rename(columns={"despesas_com_pessoal": "antes"})
    .assign(variavel="despesas_com_pessoal")
)

amostra_outras = (
    dados.loc[
        dados["outras_despesas_operacionais"].str.contains(",", regex=False, na=False),
        ["id_registro", "outras_despesas_operacionais"]
    ]
    .head(4)
    .rename(columns={"outras_despesas_operacionais": "antes"})
    .assign(variavel="outras_despesas_operacionais")
)

amostra_formatos = pd.concat(
    [amostra_pessoal, amostra_outras],
    ignore_index=True
)[["id_registro", "variavel", "antes"]]

amostra_formatos["ordem_amostra"] = range(len(amostra_formatos))

display(amostra_formatos.drop(columns="ordem_amostra"))


### Padronize a representação

Nesta atividade, o padrão adotado para as despesas será: **sem símbolo monetário e com ponto como separador decimal**.

Execute a célula seguinte para uniformizar a escrita dos registros equivalentes.


In [ ]:
def padronizar_formato_despesa(serie):
    texto = serie.astype("string").str.strip().str.replace("R$", "", regex=False).str.strip()
    tem_virgula = texto.str.contains(",", regex=False, na=False)
    texto.loc[tem_virgula] = (
        texto.loc[tem_virgula]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    return texto

dados_padronizados = dados.copy()
dados_padronizados["despesas_com_pessoal"] = padronizar_formato_despesa(
    dados_padronizados["despesas_com_pessoal"]
)
dados_padronizados["outras_despesas_operacionais"] = padronizar_formato_despesa(
    dados_padronizados["outras_despesas_operacionais"]
)

print("Padrão aplicado: sem símbolo monetário e com ponto como separador decimal.")


### Confira o que mudou

Compare os mesmos registros antes e depois da padronização e verifique se a quantidade de valores preenchidos foi preservada.


In [ ]:
ids_amostra = amostra_formatos["id_registro"].unique()

depois_longo = (
    dados_padronizados.loc[
        dados_padronizados["id_registro"].isin(ids_amostra),
        ["id_registro", "despesas_com_pessoal", "outras_despesas_operacionais"]
    ]
    .melt(
        id_vars="id_registro",
        var_name="variavel",
        value_name="depois"
    )
)

comparacao_formatos = (
    amostra_formatos
    .merge(depois_longo, on=["id_registro", "variavel"], how="left", validate="one_to_one")
    .sort_values("ordem_amostra")
    [["id_registro", "variavel", "antes", "depois"]]
)

display(comparacao_formatos)

conferencia_formatos = pd.DataFrame({
    "variavel": ["despesas_com_pessoal", "outras_despesas_operacionais"],
    "preenchidos_antes": [
        dados["despesas_com_pessoal"].notna().sum(),
        dados["outras_despesas_operacionais"].notna().sum(),
    ],
    "preenchidos_depois": [
        dados_padronizados["despesas_com_pessoal"].notna().sum(),
        dados_padronizados["outras_despesas_operacionais"].notna().sum(),
    ],
})

display(conferencia_formatos)


### Registre — Situação 1

No caderno, responda em poucas linhas:

**Qual padrão foi adotado para representar os valores de despesas? Que evidência do notebook mostra que a padronização preservou as informações preenchidas?**


# Situação 2 — Rótulos diferentes para a mesma categoria

Agora trabalharemos com um **pequeno recorte didático adaptado**. Ele representa registros de canais de divulgação recebidos de fontes diferentes.

Antes de executar qualquer padronização, observe os rótulos e identifique quais deles representam de fato a **mesma categoria**.


In [ ]:
categorias = pd.DataFrame({
    "registro": ["C1", "C2", "C3", "C4", "C5", "C6"],
    "canal_divulgacao": [
        "WhatsApp",
        "whatsapp",
        " WHATSAPP ",
        "Redes sociais",
        "redes sociais",
        "Mídia local",
    ],
})

display(categorias)


### Decida antes de executar

- Quais rótulos identificam a mesma categoria?
- Qual rótulo único você adotaria para cada grupo equivalente?

Depois da decisão, execute a célula seguinte para aplicar os rótulos definidos para esta atividade.


In [ ]:
mapa_rotulos = {
    "WhatsApp": "WhatsApp",
    "whatsapp": "WhatsApp",
    " WHATSAPP ": "WhatsApp",
    "Redes sociais": "Redes sociais",
    "redes sociais": "Redes sociais",
    "Mídia local": "Mídia local",
}

categorias_padronizadas = categorias.copy()
categorias_padronizadas["canal_divulgacao"] = categorias_padronizadas["canal_divulgacao"].map(mapa_rotulos)

display(categorias_padronizadas)

print("Rótulos antes:", categorias["canal_divulgacao"].tolist())
print("Categorias após a padronização:", categorias_padronizadas["canal_divulgacao"].unique().tolist())


### Registre — Situação 2

No caderno, anote:

**Que critério permitiu decidir quais rótulos poderiam ser unificados e quais deveriam permanecer separados?**


# Situação 3 — Um valor incorporado ao texto

Neste recorte didático, os valores de receita já existem, mas estão incorporados a um campo de texto. Para usá-los de forma consistente, vamos colocá-los em uma variável própria.


In [ ]:
estrutura = pd.DataFrame({
    "loja": ["L1", "L2", "L3"],
    "informacao": ["Receita: 1200", "Receita: 1350", "Receita: 980"],
})

print("Antes")
display(estrutura)


In [ ]:
estrutura_organizada = estrutura.copy()
estrutura_organizada["receita"] = pd.to_numeric(
    estrutura_organizada["informacao"].str.extract(r"([0-9]+(?:[.,][0-9]+)?)")[0].str.replace(",", ".", regex=False)
)
estrutura_organizada = estrutura_organizada[["loja", "receita"]]

print("Depois")
display(estrutura_organizada)


In [ ]:
valores_antes = pd.to_numeric(
    estrutura["informacao"].str.extract(r"([0-9]+(?:[.,][0-9]+)?)")[0].str.replace(",", ".", regex=False)
)

print("Valores preservados:", valores_antes.tolist() == estrutura_organizada["receita"].tolist())


### Registre — Situação 3

No caderno, responda:

**Ao criar a variável `receita`, foram criados novos valores ou os valores da receita já existentes foram apenas reorganizados? Como o notebook permite verificar isso?**


# Conferência final

Antes de encerrar, confira se suas três decisões atendem aos critérios da atividade:

- o padrão adotado é coerente com o uso dos dados;
- registros equivalentes mantêm o mesmo significado;
- o resultado foi conferido depois da alteração.

**Produto da atividade:** os resultados padronizados/organizados no notebook e três registros curtos no caderno, um para cada situação.


---

# Expectativa de respostas e conferência

## Organização sugerida para os 26 minutos

- **3 min:** apresentação da situação, abertura do notebook e orientação sobre os três registros no caderno;
- **7 min:** Situação 1 — formatos;
- **7 min:** Situação 2 — categorias equivalentes;
- **6 min:** Situação 3 — organização estrutural;
- **3 min:** conferência coletiva dos critérios e fechamento.

A divisão é uma referência de condução. O foco deve permanecer nas decisões e verificações, não na velocidade de execução.

## Expectativa — Situação 1

Os registros de despesas podem representar a mesma informação em formas diferentes, como `R$ 667,11`, `345,58` e valores já registrados com ponto decimal. O padrão aplicado nesta atividade remove o símbolo monetário quando presente e utiliza ponto como separador decimal, mantendo a informação representada.

A resposta do estudante deve identificar o padrão adotado e usar a comparação antes/depois como evidência de preservação. A conferência adicional deve mostrar que a quantidade de valores preenchidos permanece a mesma: 4.964 em `despesas_com_pessoal` e 4.958 em `outras_despesas_operacionais`, antes e depois.

## Expectativa — Situação 2

Os três rótulos `WhatsApp`, `whatsapp` e ` WHATSAPP ` representam a mesma categoria e são unificados como `WhatsApp`. `Redes sociais` e `redes sociais` também são equivalentes entre si e são unificados como `Redes sociais`. `Mídia local` permanece distinta.

O critério esperado é **equivalência de significado**, não mera semelhança textual. Categorias com significados diferentes não devem ser fundidas.

## Expectativa — Situação 3

Não foram criados novos valores de receita. Os valores `1200`, `1350` e `980` já estavam presentes no texto e foram separados para ocupar a variável `receita`. A conferência imprime `Valores preservados: True`.

## Critérios de aceitação

Aceite formulações semanticamente equivalentes quando o estudante:

1. identifica um padrão coerente para o uso dos dados;
2. distingue equivalência de rótulos de criação ou fusão de categorias;
3. reconhece que a organização estrutural pode criar uma variável na tabela sem criar novos valores;
4. usa alguma evidência do antes/depois para justificar que a consistência foi preservada.
